In [1]:
import pandas as pd

DATA = "../data/"

debt    = pd.read_csv(DATA + "IDS_ALLCountries_Data.csv", encoding="latin-1")
country = pd.read_csv(DATA + "IDS_CountryMetaData.csv", encoding="latin-1")
series  = pd.read_csv(DATA + "IDS_SeriesMetaData.csv", encoding="latin-1")

print("Start:", debt.shape)

Start: (62983, 39)


In [8]:
debt["Country Name"].sample(50)





36173                                             Maldives
52521                       St. Vincent and the Grenadines
60909                                              Vanuatu
20139                                              Georgia
22525                                        Guinea-Bissau
38303    Middle East, North Africa, Afghanistan & Pakis...
44978                                             Pakistan
59963                                  Upper middle income
40951                                              Morocco
49993                                      Solomon Islands
15776                                     Egypt, Arab Rep.
21604                                            Guatemala
44637                                             Pakistan
50328                                   Somalia, Fed. Rep.
52681                       St. Vincent and the Grenadines
30631    Latin America & Caribbean (excluding high income)
15395                                              Ecuad

In [3]:
debt = debt.dropna(subset=["Country Code"])
print("After removing junk rows:", debt.shape)

After removing junk rows: (62978, 39)


In [9]:
debt["Country Code"] = debt["Country Code"].str.strip()
debt["Country Name"] = debt["Country Name"].str.strip()

print(repr(debt["Country Code"].iloc[0]))

'AFG'


In [10]:
real_countries = country[country["Region"].notna()]

debt = debt[debt["Country Code"].isin(real_countries["Code"])]

print("Countries left:", debt["Country Code"].nunique())
print("Rows left:", debt.shape)

Countries left: 120
Rows left: (55049, 39)


In [11]:
debt = debt.drop(columns=["Counterpart-Area Name", "Counterpart-Area Code"])
print(debt.columns.tolist()[:6])

['Country Name', 'Country Code', 'Series Name', 'Series Code', '2000', '2001']


In [12]:
year_cols = [c for c in debt.columns if c.isdigit()]   # '2000' ... '2032'

debt_long = debt.melt(
    id_vars=["Country Name", "Country Code", "Series Name", "Series Code"],
    value_vars=year_cols,
    var_name="year",
    value_name="value",
)
debt_long["year"] = debt_long["year"].astype(int)

print("Long format:", debt_long.shape)
debt_long.head()

Long format: (1816617, 6)


,Country Name,Country Code,Series Name,Series Code,year,value
0,Afghanistan,AFG,Average grace period on new external debt comm...,DT.GPA.DPPG,2000,NaN
1,Afghanistan,AFG,Average grace period on new external debt comm...,DT.GPA.OFFT,2000,NaN
2,Afghanistan,AFG,Average grace period on new external debt comm...,DT.GPA.PRVT,2000,NaN
3,Afghanistan,AFG,Average grant element on new external debt com...,DT.GRE.DPPG,2000,NaN
4,Afghanistan,AFG,Average grant element on new external debt com...,DT.GRE.OFFT,2000,NaN


In [15]:
debt_long.info()




<class 'pandas.DataFrame'>
RangeIndex: 1816617 entries, 0 to 1816616
Data columns (total 6 columns):
 #   Column        Dtype  
---  ------        -----  
 0   Country Name  str    
 1   Country Code  str    
 2   Series Name   str    
 3   Series Code   str    
 4   year          int64  
 5   value         float64
dtypes: float64(1), int64(1), str(4)
memory usage: 213.8 MB


In [16]:
debt_long = debt_long.dropna(subset=["value"])
print("After dropping empty values:", debt_long.shape)

debt_long = debt_long[debt_long["year"] <= 2024]
print("After keeping 2000-2024:", debt_long.shape)

After dropping empty values: (1142836, 6)
After keeping 2000-2024: (1060074, 6)


In [17]:
before = len(debt_long)
debt_long = debt_long.drop_duplicates(subset=["Country Code", "Series Code", "year"])
print("Duplicates removed:", before - len(debt_long))

Duplicates removed: 0


In [18]:
# 1) Countries
countries = (real_countries[["Code", "Long Name", "Region", "Income Group"]]
             .rename(columns={"Code": "country_code", "Long Name": "country_name",
                              "Region": "region", "Income Group": "income_group"}))
countries["income_group"] = countries["income_group"].fillna("Unclassified")

# 2) Indicators: take names from the debt data itself, then add the topic
indicators = (debt_long[["Series Code", "Series Name"]]
              .drop_duplicates()
              .rename(columns={"Series Code": "indicator_code",
                               "Series Name": "indicator_name"}))

topics = series[["Code", "Topic"]].rename(columns={"Code": "indicator_code",
                                                    "Topic": "topic"})
indicators = indicators.merge(topics, on="indicator_code", how="left")
indicators["topic"] = indicators["topic"].fillna("Unknown")

# Flag indicators measured in US dollars (needed for the debt questions later)
indicators["is_usd"] = indicators["indicator_name"].str.contains(
    "current US$", regex=False).astype(int)

# 3) Debt data (the main table)
debt_data = (debt_long[["Country Code", "Series Code", "year", "value"]]
             .rename(columns={"Country Code": "country_code",
                              "Series Code": "indicator_code"}))

print(len(countries), len(indicators), len(debt_data))

120 576 1060074


In [19]:
print("Unique indicator codes:", indicators["indicator_code"].is_unique)
print("US$ indicators:", indicators["is_usd"].sum())
print("Years:", debt_data["year"].min(), "to", debt_data["year"].max())
print("Countries in debt_data:", debt_data["country_code"].nunique())
print("Missing values left:\n", debt_data.isna().sum())

Unique indicator codes: True
US$ indicators: 538
Years: 2000 to 2024
Countries in debt_data: 120
Missing values left:
 country_code      0
indicator_code    0
year              0
value             0
dtype: int64


In [20]:
countries.to_csv(DATA + "clean_countries.csv", index=False)
indicators.to_csv(DATA + "clean_indicators.csv", index=False)
debt_data.to_csv(DATA + "clean_debt_data.csv", index=False)

print("Saved.")

Saved.
